In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns


TX = daily maximum temperature
TX = daily minimum temperature

In [6]:
tx=pd.read_csv('../data/raw/TX.csv', sep=';')
tn=pd.read_csv('../data/raw/TN.csv', sep=';')

In [7]:
tx.shape

(17167, 4)

In [8]:
tn.shape

(17167, 4)

In [9]:
display(tx.head())
display(tn.head())

,SOUID,DATE,TX,Q_TX
0,105838,19790101,23,0
1,105838,19790102,16,0
2,105838,19790103,13,0
3,105838,19790104,-3,0
4,105838,19790105,56,0


,SOUID,DATE,TN,Q_TN
0,105760,19790101,-75,0
1,105760,19790102,-75,0
2,105760,19790103,-72,0
3,105760,19790104,-65,0
4,105760,19790105,-14,0


In [10]:
tx.describe()

,SOUID,DATE,TX,Q_TX
count,17167.0,1.716700e+04,17167.000000,17167.0
mean,105838.0,2.002067e+07,155.227180,0.0
std,0.0,1.356527e+05,65.825219,0.0
min,105838.0,1.979010e+07,-62.000000,0.0
25%,105838.0,1.990100e+07,106.000000,0.0
50%,105838.0,2.002070e+07,152.000000,0.0
75%,105838.0,2.014040e+07,204.000000,0.0
max,105838.0,2.025123e+07,402.000000,0.0


In [11]:
tn.describe()

,SOUID,DATE,TN,Q_TN
count,17167.0,1.716700e+04,17167.000000,17167.0
mean,105760.0,2.002067e+07,76.438749,0.0
std,0.0,1.356527e+05,53.329792,0.0
min,105760.0,1.979010e+07,-118.000000,0.0
25%,105760.0,1.990100e+07,36.000000,0.0
50%,105760.0,2.002070e+07,79.000000,0.0
75%,105760.0,2.014040e+07,118.000000,0.0
max,105760.0,2.025123e+07,223.000000,0.0


In [12]:
tx.isna().sum()

SOUID      0
   DATE    0
  TX       0
Q_TX       0
dtype: int64

In [13]:
tn.isna().sum()

SOUID      0
   DATE    0
  TN       0
Q_TN       0
dtype: int64

In [17]:
tx.columns = tx.columns.str.strip()
tn.columns = tn.columns.str.strip()

In [95]:
temp=pd.merge(tx[["DATE", "TX", "Q_TX"]],tn[["DATE", "TN", "Q_TN"]],on="DATE",how="inner")

# 1. CLIMPACT MADNESS

In [92]:
rr=pd.read_csv('../data/raw/RR.csv', sep=';')

In [93]:
rr.columns = rr.columns.str.strip()

In [94]:
rr.Q_RR.value_counts()

Q_RR
0    17153
9       14
Name: count, dtype: int64

In [96]:
heathrow=pd.merge(temp,rr[["DATE", "RR", "Q_RR"]],on="DATE",how="inner")

In [97]:
heathrow.shape

(17167, 7)

In [98]:
heathrow.head()

,DATE,TX,Q_TX,TN,Q_TN,RR,Q_RR
0,19790101,23,0,-75,0,4,0
1,19790102,16,0,-75,0,0,0
2,19790103,13,0,-72,0,0,0
3,19790104,-3,0,-65,0,0,0
4,19790105,56,0,-14,0,0,0


In [99]:
heathrow=heathrow.drop(columns=["Q_TX", "Q_TN"])

In [100]:
heathrow.Q_RR.value_counts()

Q_RR
0    17153
9       14
Name: count, dtype: int64

In [101]:
heathrow.isna().sum()

DATE    0
TX      0
TN      0
RR      0
Q_RR    0
dtype: int64

In [102]:
heathrow=heathrow.drop(columns=["Q_RR"])

In [103]:
heathrow["DATE"] = pd.to_datetime(heathrow["DATE"].astype(str), format="%Y%m%d")
heathrow["TX"] = heathrow["TX"] / 10.0   
heathrow["TN"] = heathrow["TN"] / 10.0

In [104]:
heathrow.dtypes

DATE    datetime64[us]
TX             float64
TN             float64
RR               int64
dtype: object

In [105]:
# converting rain precipitaion to mm and using the name required by climpact:
heathrow['PR']=heathrow["RR"] / 10

In [106]:
heathrow.head()

,DATE,TX,TN,RR,PR
0,1979-01-01,2.3,-7.5,4,0.4
1,1979-01-02,1.6,-7.5,0,0.0
2,1979-01-03,1.3,-7.2,0,0.0
3,1979-01-04,-0.3,-6.5,0,0.0
4,1979-01-05,5.6,-1.4,0,0.0


In [107]:
heathrow=heathrow.drop(columns=["RR"])

In [108]:
heathrow.head()

,DATE,TX,TN,PR
0,1979-01-01,2.3,-7.5,0.4
1,1979-01-02,1.6,-7.5,0.0
2,1979-01-03,1.3,-7.2,0.0
3,1979-01-04,-0.3,-6.5,0.0
4,1979-01-05,5.6,-1.4,0.0


In [109]:
heathrow["year"]= heathrow["DATE"].dt.year
heathrow["month"]= heathrow["DATE"].dt.month
heathrow["day"]= heathrow["DATE"].dt.day   
heathrow.head()

,DATE,TX,TN,PR,year,month,day
0,1979-01-01,2.3,-7.5,0.4,1979,1,1
1,1979-01-02,1.6,-7.5,0.0,1979,1,2
2,1979-01-03,1.3,-7.2,0.0,1979,1,3
3,1979-01-04,-0.3,-6.5,0.0,1979,1,4
4,1979-01-05,5.6,-1.4,0.0,1979,1,5


In [110]:
heathrow=heathrow.drop(columns=["DATE"])
heathrow.head()

,TX,TN,PR,year,month,day
0,2.3,-7.5,0.4,1979,1,1
1,1.6,-7.5,0.0,1979,1,2
2,1.3,-7.2,0.0,1979,1,3
3,-0.3,-6.5,0.0,1979,1,4
4,5.6,-1.4,0.0,1979,1,5


In [111]:
heathrow.dtypes

TX       float64
TN       float64
PR       float64
year       int32
month      int32
day        int32
dtype: object

In [112]:
heathrow = heathrow.rename(columns={"year": "Year","month": "Month","day": "Day"})
heathrow = heathrow[["Year", "Month", "Day", "PR", "TN", "TX"]]
heathrow.head()

,Year,Month,Day,PR,TN,TX
0,1979,1,1,0.4,-7.5,2.3
1,1979,1,2,0.0,-7.5,1.6
2,1979,1,3,0.0,-7.2,1.3
3,1979,1,4,0.0,-6.5,-0.3
4,1979,1,5,0.0,-1.4,5.6


In [113]:
# Filtering dates:
# heathrow = heathrow[(heathrow["Year"] >= 1991) & (heathrow["Year"] <= 2020)].copy()

In [114]:
heathrow.shape

(17167, 6)

In [115]:
heathrow = heathrow.replace(-999.9, -99.9)

In [116]:
heathrow.to_csv("/Users/iirene/Desktop/heathrow.txt",index=False, header=False, sep="\t")

# 2. BASELINE CALCULATION

In [117]:
temp.head()

,DATE,TX,Q_TX,TN,Q_TN
0,19790101,23,0,-75,0
1,19790102,16,0,-75,0
2,19790103,13,0,-72,0
3,19790104,-3,0,-65,0
4,19790105,56,0,-14,0


In [118]:
# converto date time and degrees celsius:
temp["DATE"] = pd.to_datetime(temp["DATE"].astype(str), format="%Y%m%d")
temp["TX"] = temp["TX"] / 10.0   
temp["TN"] = temp["TN"] / 10.0

In [119]:
temp["Q_TX"].value_counts()

Q_TX
0    17167
Name: count, dtype: int64

In [120]:
temp["Q_TN"].value_counts()

Q_TN
0    17167
Name: count, dtype: int64

In [121]:
temp=temp.drop(columns=["Q_TX", "Q_TN"])

In [122]:
temp=temp.sort_values("DATE").reset_index(drop=True)

In [123]:
temp.head()

,DATE,TX,TN
0,1979-01-01,2.3,-7.5
1,1979-01-02,1.6,-7.5
2,1979-01-03,1.3,-7.2
3,1979-01-04,-0.3,-6.5
4,1979-01-05,5.6,-1.4


In [125]:
temp["year"]= temp["DATE"].dt.year
temp["month"]= temp["DATE"].dt.month
temp["day"]= temp["DATE"].dt.day  

In [127]:
#Dropping the 29th of february:

temp = temp.drop(temp[(temp["month"] == 2) & (temp["day"] == 29)].index)

In [129]:
temp["doy"]   = temp["DATE"].dt.dayofyear

print(temp.head())
print(f"\nTotal rows: {len(temp)}  |  Date range: {temp['DATE'].min().date()} → {temp['DATE'].max().date()}")

        DATE   TX   TN  year  month  day  doy
0 1979-01-01  2.3 -7.5  1979      1    1    1
1 1979-01-02  1.6 -7.5  1979      1    2    2
2 1979-01-03  1.3 -7.2  1979      1    3    3
3 1979-01-04 -0.3 -6.5  1979      1    4    4
4 1979-01-05  5.6 -1.4  1979      1    5    5

Total rows: 17155  |  Date range: 1979-01-01 → 2025-12-31


In [134]:
print((temp[['TX','TN']] < -99).any())

TX    False
TN    False
dtype: bool


In [135]:
#chossing the 30 years from which to elaborate the baseline:
baseline = temp[(temp["year"] >= 1991) & (temp["year"] <= 2020)].copy()

In [136]:
print(f"Baseline rows: {len(baseline)}")

Baseline rows: 10950


In [137]:
baseline.isna().sum()

DATE     0
TX       0
TN       0
year     0
month    0
day      0
doy      0
dtype: int64

In [141]:
# establishing window for calculation of average temperature of each day (-5, +5 days)
window=5

In [144]:
# variables where we will store the 90th percentile min and max temperatures for each day of the year:
tx_p90={}
tn_p90={}

In [145]:
# loop to calculate the 90thp min or max temperature for each day of the year based on a 5 day window:

for doy in range(1, 366):
    window_doys = [(doy + d - 1) % 365 + 1 for d in range(-window, window + 1)]
    subset = baseline[baseline["doy"].isin(window_doys)]
    tx_vals = subset["TX"].dropna()
    tn_vals = subset["TN"].dropna()
    tx_p90[doy] = np.percentile(tx_vals, 90)
    tn_p90[doy] = np.percentile(tn_vals, 90)
    

In [147]:
# convert to pd series for mapping
tx_p90_series = pd.Series(tx_p90)
tn_p90_series = pd.Series(tn_p90)

In [149]:
print("Sample thresholds around mid-July (DOY ~196):")
for d in range(193, 200):
    print(f"  DOY {d}: TX_p90={tx_p90[d]:.2f}°C | TN_p90={tn_p90[d]:.2f}°C")

Sample thresholds around mid-July (DOY ~196):
  DOY 193: TX_p90=28.21°C | TN_p90=16.81°C
  DOY 194: TX_p90=28.50°C | TN_p90=16.90°C
  DOY 195: TX_p90=28.52°C | TN_p90=16.91°C
  DOY 196: TX_p90=28.72°C | TN_p90=17.00°C
  DOY 197: TX_p90=28.91°C | TN_p90=17.10°C
  DOY 198: TX_p90=29.31°C | TN_p90=17.01°C
  DOY 199: TX_p90=29.51°C | TN_p90=17.10°C


# 3. APPLY THRESHOLDS TO DATASET

In [155]:
evaluation=temp.copy()

In [156]:
evaluation.head()

,DATE,TX,TN,year,month,day,doy
0,1979-01-01,2.3,-7.5,1979,1,1,1
1,1979-01-02,1.6,-7.5,1979,1,2,2
2,1979-01-03,1.3,-7.2,1979,1,3,3
3,1979-01-04,-0.3,-6.5,1979,1,4,4
4,1979-01-05,5.6,-1.4,1979,1,5,5


In [157]:
# create column with threshold temperatures (by doy):

evaluation["TX_p90"] = evaluation["doy"].map(tx_p90_series)
evaluation["TN_p90"] = evaluation["doy"].map(tn_p90_series)

In [158]:
# flag temperatures above the threshhold (by doy):
evaluation["above_TX_p90"] = evaluation["TX"] > evaluation["TX_p90"]
evaluation["above_TN_p90"] = evaluation["TN"] > evaluation["TN_p90"]

In [159]:
# flag days with both min and max temperatures above the threshold (by doy):
evaluation["above_baseline_day"] = (evaluation["above_TX_p90"] & evaluation["above_TN_p90"])

In [162]:
evaluation.groupby("year")["above_TX_p90"].value_counts()

year  above_TX_p90
1979  False           347
      True             18
1980  False           349
      True             16
1981  False           349
                     ... 
2023  True             60
2024  False           312
      True             53
2025  False           297
      True             68
Name: count, Length: 94, dtype: int64

In [163]:
evaluation.head()

,DATE,TX,TN,year,month,day,doy,TX_p90,TN_p90,above_TX_p90,above_TN_p90,above_baseline_day
0,1979-01-01,2.3,-7.5,1979,1,1,1,12.32,7.40,False,False,False
1,1979-01-02,1.6,-7.5,1979,1,2,2,12.50,7.21,False,False,False
2,1979-01-03,1.3,-7.2,1979,1,3,3,12.50,7.31,False,False,False
3,1979-01-04,-0.3,-6.5,1979,1,4,4,12.51,7.50,False,False,False
4,1979-01-05,5.6,-1.4,1979,1,5,5,12.50,7.50,False,False,False


In [168]:
# function to identify heatwaves where we create a group everytime the value in hot_col changes and sums the number of occurences. 
# second we count the number of rows in each group and store it in run_length variable
# Then we filter by those that have true in the hot col and 

def identify_heatwaves(df, hot_col, min_duration=3):
    groups = (df[hot_col] != df[hot_col].shift()).cumsum()
    run_length = df.groupby(groups)[hot_col].transform("size")
    df["heatwave"] = ((df[hot_col] == True)& (run_length >= min_duration))
    return df

In [169]:
# apply function and see result:
evaluation = identify_heatwaves(evaluation,hot_col="above_TX_p90",min_duration=3)

In [170]:
evaluation.head()

,DATE,TX,TN,year,month,day,doy,TX_p90,TN_p90,above_TX_p90,above_TN_p90,above_baseline_day,heatwave
0,1979-01-01,2.3,-7.5,1979,1,1,1,12.32,7.40,False,False,False,False
1,1979-01-02,1.6,-7.5,1979,1,2,2,12.50,7.21,False,False,False,False
2,1979-01-03,1.3,-7.2,1979,1,3,3,12.50,7.31,False,False,False,False
3,1979-01-04,-0.3,-6.5,1979,1,4,4,12.51,7.50,False,False,False,False
4,1979-01-05,5.6,-1.4,1979,1,5,5,12.50,7.50,False,False,False,False


In [173]:
# days in heatwave per year:
evaluation.groupby("year")["heatwave"].sum()

year
1979     7
1980     9
1981    10
1982    10
1983    12
1984    15
1985    14
1986    11
1987     7
1988     0
1989    28
1990    32
1991     6
1992    12
1993     4
1994    10
1995    34
1996     6
1997    21
1998    17
1999    12
2000     9
2001     6
2002     6
2003    30
2004    11
2005     8
2006    24
2007     9
2008    14
2009    12
2010     4
2011    23
2012    21
2013    10
2014    21
2015    22
2016    20
2017    17
2018    37
2019    26
2020    37
2021    23
2022    36
2023    32
2024    20
2025    45
Name: heatwave, dtype: int64

In [181]:
# the heatwave column gives us the days in heatwave per year NOT number of heatwaves per year, 
# so we'll create a new column counting everytime a heatwave starts so then we can sum per year and get the actuall number of heatwaves per year:

evaluation["heatwave_start"] = ((evaluation["heatwave"]==True) & (evaluation["heatwave"].astype(int).diff() > 0))
evaluation.head()

,DATE,TX,TN,year,month,day,doy,TX_p90,TN_p90,above_TX_p90,above_TN_p90,above_baseline_day,heatwave,heatwave_start
0,1979-01-01,2.3,-7.5,1979,1,1,1,12.32,7.40,False,False,False,False,False
1,1979-01-02,1.6,-7.5,1979,1,2,2,12.50,7.21,False,False,False,False,False
2,1979-01-03,1.3,-7.2,1979,1,3,3,12.50,7.31,False,False,False,False,False
3,1979-01-04,-0.3,-6.5,1979,1,4,4,12.51,7.50,False,False,False,False,False
4,1979-01-05,5.6,-1.4,1979,1,5,5,12.50,7.50,False,False,False,False,False


In [182]:
evaluation.groupby("year")["heatwave_start"].sum()

year
1979     2
1980     3
1981     3
1982     3
1983     3
1984     4
1985     3
1986     3
1987     2
1988     0
1989     5
1990     8
1991     2
1992     2
1993     1
1994     3
1995     7
1996     2
1997     7
1998     4
1999     3
2000     3
2001     2
2002     2
2003     7
2004     3
2005     2
2006     5
2007     3
2008     3
2009     4
2010     1
2011     4
2012     5
2013     2
2014     6
2015     5
2016     5
2017     5
2018     9
2019     5
2020     9
2021     6
2022     8
2023     5
2024     5
2025    10
Name: heatwave_start, dtype: int64

In [178]:
evaluation.groupby("year")["above_TX_p90"].sum()

year
1979    18
1980    16
1981    16
1982    21
1983    29
1984    27
1985    23
1986    16
1987    21
1988    15
1989    55
1990    49
1991    18
1992    26
1993    18
1994    27
1995    52
1996    21
1997    43
1998    34
1999    26
2000    23
2001    27
2002    29
2003    51
2004    22
2005    43
2006    48
2007    32
2008    24
2009    26
2010    19
2011    55
2012    35
2013    25
2014    43
2015    43
2016    38
2017    40
2018    64
2019    39
2020    64
2021    43
2022    61
2023    60
2024    53
2025    68
Name: above_TX_p90, dtype: int64

In [184]:
# adding the may-sep condition to see if there's a differince and if it's negligible:
evaluation["heatwave_summer"] = (evaluation["heatwave"] & evaluation["month"].between(5, 9))
evaluation.groupby("year")["heatwave_summer"].sum()

year
1979     0
1980     3
1981     3
1982    10
1983     6
1984     7
1985     6
1986     4
1987     0
1988     0
1989    25
1990    16
1991     3
1992     8
1993     4
1994     0
1995    27
1996     6
1997     8
1998     4
1999     8
2000     3
2001     6
2002     0
2003    20
2004     6
2005     4
2006    21
2007     0
2008     6
2009     6
2010     4
2011     4
2012    13
2013    10
2014     2
2015     0
2016    13
2017    14
2018    24
2019     9
2020    25
2021    16
2022    19
2023    19
2024     7
2025    19
Name: heatwave_summer, dtype: int64

In [185]:
# number of summer heatwaves filtered may-sep:
evaluation["heatwave_start_summer"] = (evaluation["heatwave_start"] & evaluation["month"].between(5, 9))
evaluation.groupby("year")["heatwave_start_summer"].sum()

year
1979    0
1980    1
1981    1
1982    3
1983    1
1984    2
1985    1
1986    1
1987    0
1988    0
1989    4
1990    3
1991    1
1992    1
1993    1
1994    0
1995    6
1996    2
1997    2
1998    1
1999    2
2000    1
2001    2
2002    0
2003    4
2004    2
2005    1
2006    4
2007    0
2008    1
2009    2
2010    1
2011    1
2012    3
2013    2
2014    1
2015    0
2016    3
2017    4
2018    6
2019    2
2020    6
2021    4
2022    4
2023    2
2024    2
2025    4
Name: heatwave_start_summer, dtype: int64